In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [2]:
df=pd.read_csv("skills2025.csv")
print(df.shape)

(65437, 114)


In [3]:
cols=["DevType", "Industry", "LanguageHaveWorkedWith", "LanguageWantToWorkWith", "LanguageAdmired", "DatabaseHaveWorkedWith", "DatabaseWantToWorkWith", "DatabaseAdmired", "PlatformHaveWorkedWith", "PlatformWantToWorkWith", "PlatformAdmired", "WebframeHaveWorkedWith", "WebframeWantToWorkWith", "WebframeAdmired", "EmbeddedHaveWorkedWith", "EmbeddedWantToWorkWith", "EmbeddedAdmired", "MiscTechHaveWorkedWith", "MiscTechWantToWorkWith", "MiscTechAdmired", "ToolsTechHaveWorkedWith", "ToolsTechWantToWorkWith", "ToolsTechAdmired", "NEWCollabToolsHaveWorkedWith", "NEWCollabToolsWantToWorkWith", "NEWCollabToolsAdmired", "OfficeStackAsyncHaveWorkedWith", "OfficeStackAsyncWantToWorkWith", "OfficeStackAsyncAdmired", "OfficeStackSyncHaveWorkedWith", "OfficeStackSyncWantToWorkWith", "OfficeStackSyncAdmired","YearsCode", "YearsCodePro", "WorkExp"]
df=df[cols]

In [4]:
df.dropna(subset=['DevType'],inplace=True)

In [5]:
null_list_temp=df.isnull().sum()
null_list_temp=pd.DataFrame(null_list_temp)
null_list_temp.reset_index(inplace=True)
null_list_temp.rename(columns={'index':'field',0:'null_entries'},inplace=True)
null_list_temp.sort_values(by=['null_entries'],inplace=True,ascending=False)

In [6]:
#add core fields that are absolutely necessary for dataset
df=df[["DevType", "LanguageHaveWorkedWith", "LanguageWantToWorkWith", "DatabaseHaveWorkedWith", "DatabaseWantToWorkWith", "YearsCode", "YearsCodePro"]]

In [7]:
df.drop(columns=["YearsCode", "YearsCodePro"],inplace=True)

In [8]:
#check if no of null entries are still within threshold currently 10.89% null rate
print((df.isnull().mean()*100).round(2))
print((df.isnull().mean().mean()*100).round(2))

DevType                    0.00
LanguageHaveWorkedWith     3.45
LanguageWantToWorkWith     9.69
DatabaseHaveWorkedWith    18.34
DatabaseWantToWorkWith    30.64
dtype: float64
12.42


In [9]:
#Dropping responses by students and renaming 'others pls specify' to 'others' in DevType
df=df[(df['DevType']!='Student') & (df['DevType']!='Other (please specify):')]

In [10]:
#dropping values where all four transaction columns were null as they are the main feature of this project.
df.dropna(axis=0,how='all',subset=['LanguageHaveWorkedWith','DatabaseHaveWorkedWith'],inplace=True)

In [11]:
#changing dataypes from object to int8 for years of exp and devtype as categories
coltran2=ColumnTransformer([('tr1',SimpleImputer(strategy='constant',fill_value=''),['LanguageHaveWorkedWith','LanguageWantToWorkWith','DatabaseHaveWorkedWith','DatabaseWantToWorkWith'])],remainder='passthrough')
df=coltran2.fit_transform(df)
df=pd.DataFrame(df)
df.rename(columns={0:'LanguageHaveWorkedWith',1:'LanguageWantToWorkWith',2:'DatabaseHaveWorkedWith',3:'DatabaseWantToWorkWith',4:'DevType'},inplace=True)
df=df[['DevType','LanguageHaveWorkedWith','LanguageWantToWorkWith','DatabaseHaveWorkedWith','DatabaseWantToWorkWith']]

In [12]:
df['DevType']=df['DevType'].astype('category')
df['WorkWithAgg']=df[['LanguageHaveWorkedWith','DatabaseHaveWorkedWith']].apply(lambda row: ';'.join(list(set((';'.join(x for x in row if x!='')).split(';')))),axis=1)
df['TotalAgg']=df[['LanguageHaveWorkedWith','LanguageWantToWorkWith', 'DatabaseHaveWorkedWith','DatabaseWantToWorkWith']].apply(lambda row: ';'.join(list(set((';'.join(x for x in row if x!='')).split(';')))),axis=1)

In [13]:
df.drop(columns=['LanguageHaveWorkedWith','LanguageWantToWorkWith','DatabaseHaveWorkedWith','DatabaseWantToWorkWith'],inplace=True)

In [14]:
#creating a list of workwithagg and totalagg where it countains all skills of each cells it is redundant as we take in all skills as items to appends in list 
# we later use value counts on both lists to get two 1-item dataset one for each
lstwork=[]
lsttotal=[]
for i in range(df.shape[0]):
    lstwork.extend(df['WorkWithAgg'][i].split(';'))
    lsttotal.extend(df['TotalAgg'][i].split(';'))

In [15]:
df.head(1)

,DevType,WorkWithAgg,TotalAgg
0,"Developer, full-stack",JavaScript;Java;Bash/Shell (all shells);Postgr...,JavaScript;Java;Bash/Shell (all shells);Kotlin...


In [16]:
# lstwork and lsttotal are our output arrays we'll now use value counts to turn it into a 1-item dataset
lstwork=np.array(lstwork)
lsttotal=np.array(lsttotal)

lstwork=pd.Series(lstwork)
lsttotal=pd.Series(lsttotal)

x=lstwork.value_counts().sort_values(ascending=False)
y=lsttotal.value_counts().sort_values(ascending=False)

lstwork=lstwork.value_counts().sort_values(ascending=False).index
lsttotal=lsttotal.value_counts().sort_values(ascending=False).index

min_count = int(0.01 * df.shape[0])
x = x[x >= min_count]
y = y[y >= min_count]
lstwork = x.index
lsttotal = y.index

In [17]:
#final columns needed by us
df['workwithnew']=''
df['totalnew']=''
df['workwithnew']=df.apply(lambda row: [str(row['DevType'])] + [x for x in lstwork if x in row['WorkWithAgg'].split(';')],axis=1)
df['totalnew']=df.apply(lambda row: [str(row['DevType'])] + [x for x in lsttotal if x in row['TotalAgg'].split(';')],axis=1)

In [18]:
#removal of the two columns we wont need and renaming the columns well need
df.drop(columns=['WorkWithAgg','TotalAgg'],inplace=True)
df.rename(columns={'workwithnew':'workwithcolumn','totalnew':'work_wantcolumn'},inplace=True)

In [19]:
print((df['workwithcolumn'].str.len() <= 1).sum(), (df['work_wantcolumn'].str.len() <= 1).sum())
df = df[df['workwithcolumn'].str.len() > 1].reset_index(drop=True)

32 0


In [23]:
df.drop(columns=['DevType'],inplace=True)

In [26]:
df.to_csv("FinalDataset.csv")